In [ ]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb lightgbm

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import wandb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

FIXED = dict(verbose=-1)

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
features = pd.read_csv(f'{DATA_DIR}/features.csv', parse_dates=['Date'])
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')

train.shape, features.shape, stores.shape

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [ ]:
MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
PRE_CHRISTMAS = pd.to_datetime(['2010-12-24', '2011-12-23', '2012-12-21'])

FEATURE_COLS = [
    'Store', 'Dept', 'IsHoliday', 'Size', 'Type',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Year', 'Month', 'Week', 'IsPreChristmas',
    'Sales_Lag52', 'Sales_CalLag1y', 'Sales_CalLag2y', 'Sales_CalLag_Mean',
    'Series_Mean', 'Series_Median', 'Series_Std', 'Series_WoyMean',
    'Dept_WoyScaled', 'Dept_Mean', 'Store_Mean',
]

# v4-is mtavari cvlileba: kalendarze gasworebuli lag-ebi. Sales_Lag52 igive kviras
# igebs sharshandeli wlidan (364 dge ukan), magram shoba yovel wels sxva dgheze modis
# da 2012-shi (test) sadghesaswaulo dgheebis nawili shemdeg kvirashi gadadis.
# Sales_CalLag1y amitom kviris 7 dghes calke uyurebs: titoeuls poulobs zustad
# 1 wlis win igive kalendarul dghes, uyurebs romel walmart-kvirashi ijda is dghe
# da shesabamisi wonebit urevs sharshandel kvirebs. shedegad spike-is gadatana
# feature-shivea chadebuli da models araferi aqvs saswavli rac monacemebshi ar chans.
class FeatureBuilderV4(BaseEstimator, TransformerMixin):
    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df
        self.features_df = features_df

    def fit(self, X, y=None):
        df = X[['Store', 'Dept', 'Date']].copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['y'] = y.values if hasattr(y, 'values') else y

        self.history_ = df.set_index(['Store', 'Dept', 'Date'])['y']
        self.series_stats_ = df.groupby(['Store', 'Dept'])['y'].agg(['mean', 'median', 'std'])
        woy = df['Date'].dt.isocalendar().week.astype(int)
        self.woy_mean_ = df.assign(woy=woy).groupby(['Store', 'Dept', 'woy'])['y'].mean()
        self.dept_mean_ = df.groupby('Dept')['y'].mean()
        self.store_mean_ = df.groupby('Store')['y'].mean()

        sm = df.groupby(['Store', 'Dept'])['y'].transform('mean')
        norm = pd.Series(np.where(sm > 0, df['y'] / sm, np.nan), index=df.index)
        self.dept_shape_ = norm.groupby([df['Dept'], woy]).mean()
        return self

    def _week_lag(self, df, weeks):
        idx = pd.MultiIndex.from_arrays(
            [df['Store'], df['Dept'], df['Date'] - pd.Timedelta(weeks=weeks)])
        return self.history_.reindex(idx).values

    def _cal_lag(self, df, years):
        pairs = {}
        for d in df['Date'].unique():
            d = pd.Timestamp(d)
            wts = {}
            for i in range(7):
                anniv = (d - pd.Timedelta(days=i)) - pd.DateOffset(years=years)
                we = anniv + pd.Timedelta(days=(4 - anniv.weekday()) % 7)
                wts[we] = wts.get(we, 0.0) + 1.0 / 7.0
            pairs[d] = sorted(wts.items())

        num = np.zeros(len(df))
        eff = np.zeros(len(df))
        max_p = max(len(v) for v in pairs.values())
        for j in range(max_p):
            we_col = df['Date'].map(lambda d, j=j: pairs[d][j][0] if len(pairs[d]) > j else pd.NaT)
            wt_col = df['Date'].map(lambda d, j=j: pairs[d][j][1] if len(pairs[d]) > j else 0.0).values
            idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept'], we_col])
            vals = self.history_.reindex(idx).values.astype(float)
            ok = ~np.isnan(vals)
            num[ok] += wt_col[ok] * vals[ok]
            eff[ok] += wt_col[ok]
        return np.where(eff > 0, num / np.maximum(eff, 1e-9), np.nan)

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.merge(self.stores_df, on='Store', how='left')
        df = df.merge(self.features_df.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

        df[MARKDOWN_COLS] = df[MARKDOWN_COLS].fillna(0)
        df['Type'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2})
        df['IsHoliday'] = df['IsHoliday'].astype(int)

        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['IsPreChristmas'] = df['Date'].isin(PRE_CHRISTMAS).astype(int)

        df['Sales_Lag52'] = self._week_lag(df, 52)
        df['Sales_CalLag1y'] = self._cal_lag(df, 1)
        df['Sales_CalLag2y'] = self._cal_lag(df, 2)
        df['Sales_CalLag_Mean'] = pd.concat(
            [df['Sales_CalLag1y'], df['Sales_CalLag2y']], axis=1).mean(axis=1).values

        pair_idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept']])
        stats = self.series_stats_.reindex(pair_idx)
        df['Series_Mean'] = stats['mean'].values
        df['Series_Median'] = stats['median'].values
        df['Series_Std'] = stats['std'].values

        woy_t = df['Date'].dt.isocalendar().week.astype(int)
        woy_idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept'], woy_t])
        df['Series_WoyMean'] = self.woy_mean_.reindex(woy_idx).values

        shape_idx = pd.MultiIndex.from_arrays([df['Dept'], woy_t])
        df['Dept_WoyScaled'] = self.dept_shape_.reindex(shape_idx).values * df['Series_Mean'].values

        df['Dept_Mean'] = self.dept_mean_.reindex(df['Dept']).values
        df['Store_Mean'] = self.store_mean_.reindex(df['Store']).values

        return df[FEATURE_COLS]

In [ ]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('LightGBM_v4_Training')

In [ ]:
with mlflow.start_run(run_name='LightGBM_v4_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v4_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    missing_markdown_pct = float(features[MARKDOWN_COLS].isna().mean().mean())

    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    mlflow.log_metric('missing_markdown_pct', missing_markdown_pct)
    wandb.log({'negative_sales_rows': n_negative, 'missing_markdown_pct': missing_markdown_pct})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

In [ ]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

# diagnostikam achvena rom 3-fold CV kaggle-is qcevas ver imeorebs: fold 2-shi
# saerto ar aris noemberi/dekemberi, testi ki zustad iq iwyeba. am holdout-s
# igive kalendaruli forma aqvs rac tests (oqtombramde vswavlobt, shemdegi
# noembridan ivlisamde vafasebt) da diagnostikashi kaggle-is qulebtan
# bevrad axlos ijda, amitom hiperparametrebs mასze vircheavt
HOLDOUT_CUTOFF = pd.Timestamp('2011-10-28')
HOLDOUT_END = pd.Timestamp('2012-07-27')

In [ ]:
SEARCH_SPACE = {
    'n_estimators': [500, 1000, 1500],
    'num_leaves': [31, 63, 127, 255],
    'learning_rate': [0.03, 0.05, 0.1],
    'min_child_samples': [20, 50, 100, 200],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}
N_TRIALS = 20

rng = np.random.default_rng(42)

def sample_params(rng):
    return {k: v[rng.integers(len(v))] for k, v in SEARCH_SPACE.items()}

tm = train['Date'] <= HOLDOUT_CUTOFF
vm = (train['Date'] > HOLDOUT_CUTOFF) & (train['Date'] <= HOLDOUT_END)

builder = FeatureBuilderV4(stores, features)
builder.fit(train.loc[tm, ['Store', 'Dept', 'Date']], y_train[tm])
X_tr, y_tr = builder.transform(train[tm]), y_train[tm]
X_val, y_val = builder.transform(train[vm]), y_train[vm]
hol = train.loc[vm, 'IsHoliday'].values

def holdout_score(params):
    model = LGBMRegressor(**params, **FIXED)
    model.fit(X_tr, y_tr)
    return wmae(y_val.values, model.predict(X_val), hol)

with mlflow.start_run(run_name='LightGBM_v4_Tuning'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v4_Tuning', reinit='finish_previous')

    best_wmae, best_params = None, None
    for t in range(N_TRIALS):
        params = sample_params(rng)
        score = holdout_score(params)
        mlflow.log_metric('trial_wmae_holdout', score, step=t)
        wandb.log({'trial': t, 'trial_wmae_holdout': score, **params})
        if best_wmae is None or score < best_wmae:
            best_wmae, best_params = score, params
        print(t, round(score, 1), params)

    mlflow.log_metric('best_wmae_holdout', best_wmae)
    mlflow.log_dict(best_params, 'best_params.json')
    wandb.log({'best_wmae_holdout': best_wmae})
    wandb.finish()

best_wmae, best_params

In [ ]:
with mlflow.start_run(run_name='LightGBM_v4_CV'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v4_CV', config=best_params, reinit='finish_previous')

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        ftm = train['Date'] <= train_end
        fvm = (train['Date'] > train_end) & (train['Date'] <= val_end)

        pipe = Pipeline([
            ('features', FeatureBuilderV4(stores, features)),
            ('model', LGBMRegressor(**best_params, **FIXED)),
        ])
        pipe.fit(train.loc[ftm, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[ftm])
        preds = pipe.predict(train.loc[fvm, ['Store', 'Dept', 'Date', 'IsHoliday']])

        score = wmae(y_train[fvm].values, preds, train.loc[fvm, 'IsHoliday'].values)
        fold_scores.append(score)
        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'fold': i, 'wmae': score})

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    mlflow.log_metric('wmae_holdout', best_wmae)
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores)),
               'wmae_holdout': best_wmae})
    wandb.finish()

fold_scores

In [ ]:
with mlflow.start_run(run_name='LightGBM_v4_Final'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v4_Final', config=best_params, reinit='finish_previous')

    pipeline = Pipeline([
        ('features', FeatureBuilderV4(stores, features)),
        ('model', LGBMRegressor(**best_params, **FIXED)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

train_wmae

In [ ]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_LightGBM_v4.ipynb in Colab"
    !git push